In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df = df.dropna(axis=1, how='all')

X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 2:], df.iloc[:, 1], test_size=0.2)

# Scale the features and encode the labels
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)


class SimpleNN(nn.Module):
  def __init__(self,num_features):
    super().__init__()
    
    self.linear = nn.Linear(num_features,1)
    self.sigmoid = nn.Sigmoid()

  def forward(self,features):
    out = self.linear(features)
    out = self.sigmoid(out)
    return out
  
model = SimpleNN(X_train_tensor.shape[1])
optimizer = torch.optim.SGD( model.parameters(), lr = 0.01 )
loss_function = nn.BCELoss()

batch_size = 32
epochs = 25
n_sample = len(X_train_tensor)

for epoch in range(epochs):

# Mini-batch gradient descent
  for start_indx in range(0,n_sample,batch_size):
    end_indx = start_indx + batch_size

    print(f" Start index {start_indx}, end indx {end_indx}\n")

    X_batch = X_train_tensor[start_indx:end_indx]
    y_batch = y_train_tensor[start_indx:end_indx]

    #forward pass
    y_pred = model(X_batch)
    loss = loss_function(y_pred,y_batch.reshape(-1,1))

    #upate step
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

  print(f" {epoch+1} , Loss = {loss.item()} ")

 Start index 0, end indx 32

 Start index 32, end indx 64

 Start index 64, end indx 96

 Start index 96, end indx 128

 Start index 128, end indx 160

 Start index 160, end indx 192

 Start index 192, end indx 224

 Start index 224, end indx 256

 Start index 256, end indx 288

 Start index 288, end indx 320

 Start index 320, end indx 352

 Start index 352, end indx 384

 Start index 384, end indx 416

 Start index 416, end indx 448

 Start index 448, end indx 480

 1 , Loss = 0.6314471364021301 
 Start index 0, end indx 32

 Start index 32, end indx 64

 Start index 64, end indx 96

 Start index 96, end indx 128

 Start index 128, end indx 160

 Start index 160, end indx 192

 Start index 192, end indx 224

 Start index 224, end indx 256

 Start index 256, end indx 288

 Start index 288, end indx 320

 Start index 320, end indx 352

 Start index 352, end indx 384

 Start index 384, end indx 416

 Start index 416, end indx 448

 Start index 448, end indx 480

 2 , Loss = 0.4782361686

What are the problems of this approach?

1. No data shuffling or randomization.
2. No abstraction in batching.
3. No parallel execution of the same task.
4. Slow for large datasets.

# Dataset & Dataloader
Dataset and Dataloader are two important components in PyTorch that help in efficiently loading and processing data for training machine learning models.\

Dataset: A Dataset is a PyTorch class that represents a collection of data samples. It provides an interface to access individual data points and their corresponding labels. Custom Dataset classes can be created by subclassing the `torch.utils.data.Dataset` class and implementing the __init__, __len__, and __getitem__ methods.\
The __init__ method initializes the dataset, the __len__ method returns the total number of samples in the dataset, and the __getitem__ method retrieves a specific sample and its label based on an index.\
Dataset is useful for handling various types of data, such as images, text, or tabular data, and allows for easy preprocessing and augmentation of data samples before they are fed into the model. For massive datasets, it can be used to load data on-the-fly, which helps in reducing memory usage and improving training efficiency.\

Dataloader: A Dataloader is a PyTorch class that provides an efficient way to load data from a Dataset. It handles batching, shuffling, and parallel loading of data. The DataLoader class is part of the `torch.utils.data` module and can be used to create an iterable over a Dataset.\
When creating a DataLoader, parameters such as batch_size (the number of samples per batch), shuffle (whether to shuffle the data at every epoch), and num_workers (the number of subprocesses to use for data loading) can be specified.\
The DataLoader will automatically handle the batching of data and can also shuffle the data at the beginning of each epoch, which helps in improving the training process by reducing overfitting and ensuring that the model sees different samples in each epoch.\
Using Dataset and DataLoader together allows for efficient data loading and processing, especially when working with large datasets, as it can significantly speed up the training process by utilizing multiple CPU cores for data loading and ensuring that the data is shuffled and batched properly.\

Data Transformation: Data transforms are operations that can be applied to the data samples in a Dataset to preprocess or augment them before they are fed into the model. In PyTorch, data transforms can be implemented using the `torchvision.transforms` module, which provides a variety of common transformations for image data, such as resizing, cropping, normalization, and data augmentation techniques like random horizontal flipping or random rotation.\
Data transforms can be applied to the data samples in a Dataset by defining a transform function and passing it to the Dataset class. This allows for on-the-fly data preprocessing and augmentation, which can help improve the generalization of the model and reduce overfitting.\
Overall, using Dataset and Dataloader in PyTorch provides a powerful and efficient way to handle data loading and preprocessing, making it easier to train machine learning models on large datasets while ensuring that the data is properly shuffled, batched, and transformed for optimal performance.

## Core DataLoader Perameters
1. batch_size: The number of samples to be loaded in each batch.
2. shuffle: Whether to shuffle the data at every epoch.
3. num_workers: The number of subprocesses to use for data loading. A higher number can speed up data loading but may also increase memory usage.
4. sampler: A custom Sampler to specify the strategy for sampling data from the Dataset.
5. collate_fn: A custom Collate Function to specify how to combine individual data samples into a batch.
6. drop_last: Whether to drop the last incomplete batch if the dataset size is not divisible by the batch size.
7. pin_memory: Whether to pin memory during data loading, which can improve performance when using GPUs.
8. timeout: The timeout value for collecting a batch from workers. If the workers do not return a batch within the specified time, a TimeoutError will be raised.
9. worker_init_fn: A function to initialize the worker processes for data loading, which can be used to set random seeds or perform other setup tasks for each worker.
10. prefetch_factor: The number of batches to prefetch in the background while the current batch is being processed. This can help improve performance by overlapping data loading and model training.
11. persistent_workers: Whether to keep worker processes alive after the initial data loading is done. This can be useful for long-running training processes to avoid the overhead of restarting workers for each epoch.
12. generator: A random number generator to be used for shuffling the data when shuffle is set to True. This allows for reproducibility of the shuffling process across different runs.

### Sampler
A Sampler in PyTorch is a component that defines the strategy for sampling data from a Dataset. It is used in conjunction with the DataLoader to specify how the data should be sampled during training.\
The Sampler class is part of the `torch.utils.data` module and can be subclassed to create custom sampling strategies.\
The Sampler class has a method called `__iter__` that returns an iterator over the indices of the data samples in the Dataset. This allows for flexible sampling strategies, such as random sampling(where indices are selected randomly), sequential sampling(where indices are selected sequentially), or even more complex strategies like stratified sampling(where samples are selected to maintain class distribution) or weighted sampling(where samples are selected based on their weights).\
Using a Sampler can be particularly useful when dealing with imbalanced datasets, where certain classes may be underrepresented. By using a custom Sampler, the model sees a balanced representation of the classes during training, which can help improve the performance of the model.\
Overall, the Sampler provides a way to control how data is sampled from a Dataset, allowing for more efficient and effective training of machine learning models in PyTorch.

```python
from torch.utils.data import Dataset, DataLoader, Sampler
class CustomSampler(Sampler):
    def __init__(self, data_source):
        self.data_source = data_source

    def __iter__(self):
        # Implement the custom sampling logic here
        indices = list(range(len(self.data_source)))
        # For example, shuffle can be applied to the indices for random sampling
        random.shuffle(indices)
        return iter(indices)

    def __len__(self):
        return len(self.data_source)
# Usage
dataset = CustomDataset()
sampler = CustomSampler(dataset)
dataloader = DataLoader(dataset, batch_size=32, sampler=sampler)
```

### Collate Function
A Collate Function in PyTorch is a function that is used to specify how to combine individual data samples into a batch when using a DataLoader. It is passed as an argument to the DataLoader and is responsible for taking a list of data samples and combining them into a single batch that can be fed into the model.\
The Collate Function takes a list of data samples as input and returns a batch of data. This function can be customized to handle different types of data, such as images, text, or tabular data. For example, when working with images, the Collate Function can be used to stack the individual images into a single tensor, while for text data, it can be used to pad sequences to ensure that they have the same length before batching.\
The Collate Function is particularly useful when working with variable-length data, such as sequences of text or images of different sizes. By defining a custom Collate Function, the data is properly processed and formatted before being fed into the model, which can help improve the training process and the performance of the model.\
Overall, the Collate Function provides a way to control how individual data samples are combined into batches, allowing for more efficient and effective training of machine learning models in PyTorch.

```python
def custom_collate_fn(batch):
    # Implement the custom collate logic here
    # For example, pad sequences to the same length or stack images into a single tensor
    batch_features = [item[0] for item in batch]
    batch_labels = [item[1] for item in batch]
    # Pad features and labels as needed
    return torch.stack(batch_features), torch.stack(batch_labels)
# Usage
dataloader = DataLoader(dataset, batch_size=32, collate_fn=custom_collate_fn)
```

In [ ]:
from sklearn.datasets import make_classification
from torch.utils.data import Dataset, DataLoader

X,y =make_classification(
    n_samples=100,  # Number of samples to generate
    n_features=5,   # Total number of features
    n_informative=3,    # Number of informative features
    n_redundant=0,  # Number of redundant features
    n_classes=2,    # Number of classes(binary classification)
    random_state=42
)

class CustomDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return self.features.shape[0]
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]
    
dataset = CustomDataset(X,y)
dataloader = DataLoader(dataset, batch_size=10, shuffle=True, sampler=None) # Here dataloader is an iterable object which will give us the batches of data when we iterate over it

# Length of the dataset
print("Length of dataset:", len(dataset))

# Get a single sample
sample_features, sample_label = dataset[0]
print("Sample features:", sample_features)
print("Sample label:", sample_label)

for batch_features, batch_labels in dataloader:
    print("Batch features shape:", batch_features.shape)
    print("Batch labels shape:", batch_labels.shape)
    break

Length of dataset: 100
Sample features: [ 0.05144816 -0.15993853 -1.16920773  0.85765962 -0.01644241]
Sample label: 0
Batch features shape: torch.Size([10, 5])
Batch labels shape: torch.Size([10])


In [9]:
# Now modify the first example to use DataLoader instead of mini-batch gradient descent

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df = df.dropna(axis=1, how='all')

X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 2:], df.iloc[:, 1], test_size=0.2)

# Scale the features and encode the labels
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

class CustomDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return self.features.shape[0]
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]
    
train_dataset = CustomDataset(X_train_tensor, y_train_tensor)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataset = CustomDataset(X_test_tensor, y_test_tensor)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

class SimpleNN(nn.Module):
  def __init__(self,num_features):
    super().__init__()
    
    self.linear = nn.Linear(num_features,1)
    self.sigmoid = nn.Sigmoid()

  def forward(self,features):
    out = self.linear(features)
    out = self.sigmoid(out)
    return out
  
model = SimpleNN(X_train_tensor.shape[1])
optimizer = torch.optim.SGD( model.parameters(), lr = 0.01 )
loss_function = nn.BCELoss()

batch_size = 32
epochs = 25
n_sample = len(X_train_tensor)

for epoch in range(epochs):

    #forward pass
    for batch_features, batch_labels in train_dataloader:
        y_pred = model(batch_features)
        loss = loss_function(y_pred, batch_labels.reshape(-1, 1))

      #update step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f" {epoch+1} , Loss = {loss.item()} ")

 1 , Loss = 0.4798803925514221 
 2 , Loss = 0.2827893793582916 
 3 , Loss = 0.3403681814670563 
 4 , Loss = 0.32207658886909485 
 5 , Loss = 0.2125614732503891 
 6 , Loss = 0.4533946216106415 
 7 , Loss = 0.16932471096515656 
 8 , Loss = 0.1316370666027069 
 9 , Loss = 0.10397640615701675 
 10 , Loss = 0.18564121425151825 
 11 , Loss = 0.19616329669952393 
 12 , Loss = 0.11823362857103348 
 13 , Loss = 0.29084330797195435 
 14 , Loss = 0.1184188574552536 
 15 , Loss = 0.4523243010044098 
 16 , Loss = 0.05034557357430458 
 17 , Loss = 0.08783192187547684 
 18 , Loss = 0.209483340382576 
 19 , Loss = 0.120742566883564 
 20 , Loss = 0.053892023861408234 
 21 , Loss = 0.10354799777269363 
 22 , Loss = 0.15523862838745117 
 23 , Loss = 0.20589332282543182 
 24 , Loss = 0.20314255356788635 
 25 , Loss = 0.11048630625009537 


In [14]:
model.eval()    # Set the model to evaluation mode
accuracy_list = []

with torch.no_grad():  # Disable gradient calculation for evaluation
    for batch_features, batch_labels in test_dataloader:
        y_pred = model(batch_features)
        predicted_labels = (y_pred > 0.5).float()  # Convert probabilities to binary labels
        accuracy = (predicted_labels == batch_labels.reshape(-1, 1)).float().mean()  # Calculate accuracy for the batch
        accuracy_list.append(accuracy.item())  # Store the accuracy for this batch

accuracy_list

[0.96875, 1.0, 0.9375, 0.8888888955116272]

In [15]:
oval_accuracy = sum(accuracy_list) / len(accuracy_list)  # Calculate overall accuracy
print(f"Overall Accuracy: {oval_accuracy:.4f}")

Overall Accuracy: 0.9488
